In [40]:
import pandas as pd
import numpy as np

# Health Indicators

## Load Data

In [41]:
def data_loader(data_type, table_name):
    path = f'../../data/{data_type}/{table_name}-eng/{table_name}.csv'
    df = pd.read_csv(path)
    return df

In [42]:
healthcare_df = data_loader('health-indicators', '13100905')

In [43]:
healthcare_df['REF_DATE'].min(), healthcare_df['REF_DATE'].max(), healthcare_df.shape

(2015, 2023, (297576, 18))

In [44]:
# Check data types
healthcare_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297576 entries, 0 to 297575
Data columns (total 18 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   REF_DATE         297576 non-null  int64  
 1   GEO              297576 non-null  object 
 2   DGUID            273600 non-null  object 
 3   Age group        297576 non-null  object 
 4   Sex              297576 non-null  object 
 5   Indicators       297576 non-null  object 
 6   Characteristics  297576 non-null  object 
 7   UOM              297576 non-null  object 
 8   UOM_ID           297576 non-null  int64  
 9   SCALAR_FACTOR    297576 non-null  object 
 10  SCALAR_ID        297576 non-null  int64  
 11  VECTOR           297576 non-null  object 
 12  COORDINATE       297576 non-null  object 
 13  VALUE            221383 non-null  float64
 14  STATUS           95734 non-null   object 
 15  SYMBOL           0 non-null       float64
 16  TERMINATED       0 non-null       floa

In [45]:
print(f"{healthcare_df.shape[0]} rows")

297576 rows


- `value` represents percentage/number of people with 'unmet' healthcare needs.
- `value` is in *'thousands'* of people if `unit` is 'Number' and in '%' if `unit` is 'Percent'.

## Rename Column Names


In [46]:
healthcare_df.columns

Index(['REF_DATE', 'GEO', 'DGUID', 'Age group', 'Sex', 'Indicators',
       'Characteristics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID',
       'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED',
       'DECIMALS'],
      dtype='object')

In [47]:
# change column names to lower case
healthcare_df.columns = healthcare_df.columns.str.lower()

In [48]:
# replace spaces with underscores
healthcare_df.columns = healthcare_df.columns.str.replace(' ', '_')

In [49]:
# rename 'sex' to 'gender' for consistency
healthcare_df = healthcare_df.rename(columns={'sex':'gender'})

In [50]:
healthcare_df.columns

Index(['ref_date', 'geo', 'dguid', 'age_group', 'gender', 'indicators',
       'characteristics', 'uom', 'uom_id', 'scalar_factor', 'scalar_id',
       'vector', 'coordinate', 'value', 'status', 'symbol', 'terminated',
       'decimals'],
      dtype='object')

## Statistics of the Data

In [51]:
# statistics of the numerical columns
healthcare_df.describe()

,ref_date,uom_id,scalar_id,value,symbol,terminated,decimals
count,297576.000000,297576.000000,297576.0,2.213830e+05,0.0,0.0,297576.000000
mean,2019.000000,229.068231,0.0,1.850812e+05,NaN,NaN,0.379264
std,2.581993,7.763277,0.0,8.826952e+05,NaN,NaN,0.485205
min,2015.000000,223.000000,0.0,-1.000000e+00,NaN,NaN,0.000000
25%,2017.000000,223.000000,0.0,2.500000e+00,NaN,NaN,0.000000
50%,2019.000000,223.000000,0.0,3.270000e+01,NaN,NaN,0.000000
75%,2021.000000,239.000000,0.0,5.000000e+04,NaN,NaN,1.000000
max,2023.000000,239.000000,0.0,2.700790e+07,NaN,NaN,1.000000


Since standard deviation is 0 and min and max are the same, it means that all values in the column are the same.

Since count is 0, it means there are no non-null values in the column.

In [52]:
# categorical columns
healthcare_df.describe(include=['object'])

,geo,dguid,age_group,gender,indicators,characteristics,uom,scalar_factor,vector,coordinate,status
count,297576,273600,297576,297576,297576,297576,297576,297576,297576,297576,95734
unique,11,10,5,3,26,8,2,1,33064,33064,4
top,Newfoundland and Labrador,2021A000210,65 years and over,Both sexes,"Perceived health, very good or excellent",Number of persons,Number,units,v1600652147,1.1.1.1.1,..
freq,27360,27360,59544,101808,11745,37620,184716,297576,9,9,44285


In [53]:
# unique values in the categorical columns
for col in ['geo','age_group','gender','characteristics', 'uom', 'status']:
    print(f'{col}: {healthcare_df[col].nunique()} unique values')
    print(healthcare_df[col].unique())
    print()

geo: 11 unique values
['Canada (excluding territories)' 'Newfoundland and Labrador'
 'Prince Edward Island' 'Nova Scotia' 'New Brunswick' 'Quebec' 'Ontario'
 'Manitoba' 'Saskatchewan' 'Alberta' 'British Columbia']

age_group: 5 unique values
['Total, 18 years and over' '18 to 34 years' '35 to 49 years'
 '50 to 64 years' '65 years and over']

gender: 3 unique values
['Both sexes' 'Males' 'Females']

characteristics: 8 unique values
['Number of persons' 'Low 95% confidence interval, number of persons'
 'High 95% confidence interval, number of persons' 'Percent'
 'Low 95% confidence interval, percent'
 'High 95% confidence interval, percent'
 'Statistically different from previous reference period'
 'Statistically different from the Canada (excluding territories) rate']

uom: 2 unique values
['Number' 'Percent']

status: 4 unique values
[nan '..' 'E' 'F' 'x']



## Drop unnecessary rows

In [54]:
healthcare_df[healthcare_df['status'] == 'E'].shape[0]/ healthcare_df.shape[0] * 100

6.566725811221335

Data flagged with 'E' takes up ~7% of the data. 

In [55]:
healthcare_df.shape[0]

297576

In [56]:
cleaned_df = healthcare_df[healthcare_df['status'].isna() | (healthcare_df['status'] == 'E')]

In [57]:
print(f"{healthcare_df.shape[0] - cleaned_df.shape[0]} rows removed. {cleaned_df.shape[0]} rows remaining")

76193 rows removed. 221383 rows remaining


For the `status` column:  
- `E` — use with caution  
- `F` — too unreliable to be published  
- `..` — not available for a specific reference period  
- `x` — suppressed to meet the confidentiality requirements of the Statistics Act

Rows with non-null values in the status column (i.e., `..`, `F`, `x`) were removed to ensure data quality. Rows with status `E` (Use with caution) were retained but flagged due to potential reliability concerns. Only data meeting strict reliability criteria was included in the final analysis.

In [58]:
cleaned_df = cleaned_df[cleaned_df['characteristics'].isin(['Percent', 'Number of persons'])]

In [59]:
cleaned_df.shape[0]

55859

## Drop unnecessary numerical columns

In [60]:
healthcare_df = cleaned_df.drop(columns=['uom_id', 'scalar_id', 'symbol', 'terminated', 'decimals'])

In [61]:
# drop unnecessary categorical columns
for col in ['dguid', 'coordinate','vector','scalar_factor','characteristics']:
    if col in healthcare_df.columns:
        healthcare_df = healthcare_df.drop(columns=[col])

In [62]:
healthcare_df.columns

Index(['ref_date', 'geo', 'age_group', 'gender', 'indicators', 'uom', 'value',
       'status'],
      dtype='object')

In [63]:
# rename 'uom' to 'unit'
healthcare_df = healthcare_df.rename(columns={'uom':'unit'})

In [64]:
healthcare_df.loc[healthcare_df['unit'] == 'Number', 'value'] = \
    healthcare_df.loc[healthcare_df['unit'] == 'Number', 'value'] / 1000
# change value 'Number' in uom to 'Number_Thousands'
healthcare_df['unit'] = healthcare_df['unit'].replace('Number', 'Number_Thousands')

## Drop Missing Values 

In [65]:
# Check NA value counts and proportions
na_counts = healthcare_df.isna().sum()
na_proportions = healthcare_df.isna().mean()
na_summary = pd.DataFrame({'count': na_counts, 'proportion': na_proportions})
na_summary

,count,proportion
ref_date,0,0.000000
geo,0,0.000000
age_group,0,0.000000
gender,0,0.000000
indicators,0,0.000000
unit,0,0.000000
value,0,0.000000
status,50022,0.895505


In [66]:
healthcare_df['status'].unique()

array([nan, 'E'], dtype=object)

In [67]:
healthcare_df[
    (healthcare_df['gender'] == 'Males') &
    (healthcare_df['geo'] == 'British Columbia') &
    (healthcare_df['unit'] == 'Percent')
]

,ref_date,geo,age_group,gender,indicators,unit,value,status
30235,2015,British Columbia,"Total, 18 years and over",Males,"Perceived health, very good or excellent",Percent,59.4,NaN
30243,2015,British Columbia,"Total, 18 years and over",Males,"Perceived health, fair or poor",Percent,12.6,NaN
30251,2015,British Columbia,"Total, 18 years and over",Males,"Perceived mental health, very good or excellent",Percent,71.0,NaN
30259,2015,British Columbia,"Total, 18 years and over",Males,"Perceived mental health, fair or poor",Percent,5.5,NaN
30267,2015,British Columbia,"Total, 18 years and over",Males,"Perceived life stress, most days quite a bit o...",Percent,19.0,NaN
...,...,...,...,...,...,...,...,...
297331,2023,British Columbia,65 years and over,Males,"Fruit and vegetable consumption, 5 times or mo...",Percent,20.3,NaN
297339,2023,British Columbia,65 years and over,Males,"Sense of belonging to local community, somewha...",Percent,71.5,NaN
297347,2023,British Columbia,65 years and over,Males,"Life satisfaction, satisfied or very satisfied",Percent,84.2,NaN
297355,2023,British Columbia,65 years and over,Males,Has a regular healthcare provider,Percent,91.5,NaN


> gender

In [68]:
healthcare_df[healthcare_df['unit']=='Percent'].groupby('gender')['value'].mean()

gender
Both sexes    33.352485
Females       33.801993
Males         33.045795
Name: value, dtype: float64

In [69]:
healthcare_df[healthcare_df['unit']=='Number_Thousands'].groupby('gender')['value'].mean()

gender
Both sexes    731.837499
Females       421.222045
Males         438.507913
Name: value, dtype: float64

> geo

In [70]:
healthcare_df[healthcare_df['unit']=='Percent'].groupby('geo')['value'].mean().sort_values(ascending=False)

geo
Prince Edward Island              36.458815
Newfoundland and Labrador         35.356025
Nova Scotia                       34.900606
New Brunswick                     34.561289
Saskatchewan                      33.679795
Manitoba                          32.994947
Alberta                           32.985087
British Columbia                  32.282743
Ontario                           32.277542
Canada (excluding territories)    31.972793
Quebec                            31.046183
Name: value, dtype: float64

In [71]:
healthcare_df[healthcare_df['unit']=='Number_Thousands'].groupby('geo')['value'].mean().sort_values(ascending=False)

geo
Canada (excluding territories)    2353.530268
Ontario                            942.792273
Quebec                             558.146494
British Columbia                   335.207800
Alberta                            293.707201
Manitoba                           100.220852
Saskatchewan                        92.201272
Nova Scotia                         87.252487
New Brunswick                       73.688960
Newfoundland and Labrador           55.583911
Prince Edward Island                20.091837
Name: value, dtype: float64

## Feature Engineering

In [72]:
fe_df = healthcare_df.copy()

In [73]:
fe_df.head()

,ref_date,geo,age_group,gender,indicators,unit,value,status
0,2015,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, very good or excellent",Number_Thousands,17134.3,NaN
3,2015,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, very good or excellent",Percent,61.1,NaN
7,2015,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, fair or poor",Number_Thousands,3167.8,NaN
10,2015,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived health, fair or poor",Percent,11.3,NaN
14,2015,Canada (excluding territories),"Total, 18 years and over",Both sexes,"Perceived mental health, very good or excellent",Number_Thousands,19683.4,NaN


In [74]:
for col in ['geo', 'indicators']:
    print(f'{col}: {healthcare_df[col].nunique()} unique values')
    print(healthcare_df[col].unique())
    print()

geo: 11 unique values
['Canada (excluding territories)' 'Newfoundland and Labrador'
 'Prince Edward Island' 'Nova Scotia' 'New Brunswick' 'Quebec' 'Ontario'
 'Manitoba' 'Saskatchewan' 'Alberta' 'British Columbia']

indicators: 26 unique values
['Perceived health, very good or excellent'
 'Perceived health, fair or poor'
 'Perceived mental health, very good or excellent'
 'Perceived mental health, fair or poor'
 'Perceived life stress, most days quite a bit or extremely stressful'
 'Body mass index, adjusted self-reported, overweight'
 'Body mass index, adjusted self-reported, obese' 'Arthritis' 'Diabetes'
 'High blood pressure' 'Mood disorder' 'Anxiety disorder'
 'Current smoker, daily or occasional' 'Current smoker, daily'
 'Heavy drinking' 'Breast milk feeding initiation'
 'Exclusive breastfeeding, at least 6 months'
 'Fruit and vegetable consumption, 5 times or more per day'
 'Sense of belonging to local community, somewhat strong or very strong'
 'Life satisfaction, satisfied or ve

In [75]:
# mapping: indicator value to target column
indicator_to_col = {
    # Perceived health
    'Perceived health, very good or excellent': 'Perceived_health',
    'Perceived health, fair or poor': 'Perceived_health',
    # Perceived mental health
    'Perceived mental health, very good or excellent': 'Perceived_mental_health',
    'Perceived mental health, fair or poor': 'Perceived_mental_health',
    # Life stress
    'Perceived life stress, most days quite a bit or extremely stressful': 'Life_stress',
    # BMI
    'Body mass index, adjusted self-reported, overweight': 'BMI',
    'Body mass index, adjusted self-reported, obese': 'BMI',
    # Chronic conditions
    'Arthritis': 'Chronic_condition',
    'Diabetes': 'Chronic_condition',
    'High blood pressure': 'Chronic_condition',
    'Mood disorder': 'Chronic_condition',
    'Anxiety disorder': 'Chronic_condition',
    # Smoking
    'Current smoker, daily or occasional': 'Smoking',
    'Current smoker, daily': 'Smoking',
    # Alcohol/drugs
    'Heavy drinking': 'Alcohol_drug_use',
    'Cannabis use, past 12 months': 'Alcohol_drug_use',
    # Immunization
    'Influenza immunization in the past 12 months': 'Immunization',
    # Breastfeeding
    'Breast milk feeding initiation': 'Breastfeeding',
    'Exclusive breastfeeding, at least 6 months': 'Breastfeeding',
}

# create empty columns for each target category
for col in set(indicator_to_col.values()):
    fe_df[col] = np.nan

# populate the relevant column per row
for i, row in fe_df.iterrows():
    indicator = row['indicators']
    col = indicator_to_col.get(indicator)
    value = indicator.split(',',1)[1] if ',' in indicator else indicator
    if col:
        fe_df.at[i, col] = value

/var/folders/8n/4krln465113bpt4x1bnwnwr80000gn/T/ipykernel_84216/847197137.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value ' very good or excellent' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  fe_df.at[i, col] = value
/var/folders/8n/4krln465113bpt4x1bnwnwr80000gn/T/ipykernel_84216/847197137.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value ' most days quite a bit or extremely stressful' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  fe_df.at[i, col] = value
/var/folders/8n/4krln465113bpt4x1bnwnwr80000gn/T/ipykernel_84216/847197137.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value ' adjusted self-reported, overweight' has dtype incompatible with float64, please explicitly cast

In [76]:
print(f"Total number of rows before: {cleaned_df.shape[0]}, after: {fe_df.shape[0]}")

Total number of rows before: 55859, after: 55859


## Further Data Cleaning

In [77]:
fe_df['BMI'].unique()

array([nan, ' adjusted self-reported, overweight',
       ' adjusted self-reported, obese'], dtype=object)

In [78]:
# split values in 'BMI' column on comma and only keep second part
fe_df['BMI'] = fe_df['BMI'].str.split(',').str[1]

## Store Final Data as Parquet

In [80]:
# save as parquet
fe_df.to_parquet('../../data/health-indicators/can_health_indicator_15-23.parquet', index=False)